# Spatio-Temporal Clustering of Extreme Agricultural Fire Detections
**Godavari River Basin | VIIRS Extreme Agricultural Fires, 2019–2024**

Converts individual extreme VIIRS fire detections into unique extreme agricultural **fire events**, by grouping detections that are spatially and temporally connected (connected-component graph clustering).

**Rules:**
- Spatial: centre-to-centre distance ≤ 600 m
- Temporal: Δdate = 0 or 1 day
- An edge exists only when **both** conditions hold; connected components (direct or indirect chains) become one `Fire_ID`

## Setup & Imports

In [ ]:

import os
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import cKDTree
from pyproj import Geod

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

# WGS84 ellipsoid for geodesic distance calculations
GEOD = Geod(ellps="WGS84")

In [ ]:
# ---- Project paths ------------------------------------------------------------
PROJECT_ROOT = Path.home() / "Documents" / "Projects" / "Fire_emission_rivers"
DATA_DIR     = PROJECT_ROOT / "data" / "Godavari_River"
EXTREME_DIR  = PROJECT_ROOT / "outputs" / "Godavari_River" / "extreme_agri_fire"
OUT_DIR      = PROJECT_ROOT / "outputs" / "Godavari_River" / "fire_events"
OUT_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = EXTREME_DIR / "extreme_agri_fire_godavari_2019_2024.csv"


# ---- Clustering parameters ----------------------------------------------------

# Final spatial connectivity criterion:
# geodesic centre-to-centre distance must be <= 600 metres
SPATIAL_THRESHOLD_M = 600

# Temporal connectivity criterion:
# detections must occur on the same day or the following day
TEMPORAL_THRESHOLD_DAYS = 1


# ---- Spatial reference --------------------------------------------------------

# Keep original VIIRS coordinates in WGS84 (EPSG:4326).
# Geodesic distances will be calculated directly from longitude/latitude.
WGS84_EPSG = 4326

# National projected CRS for India.
# This may be used later only for efficient candidate-neighbour searching,
# NOT for the final 600 m distance decision.
INDIA_CRS_EPSG = 7755

## 1. Load Stage 5 extreme-detection dataset

In [ ]:
df = pd.read_csv(INPUT_PATH)

# Standardize date/time fields
df["acq_datetime"] = pd.to_datetime(df["acq_datetime"])
df["acq_date"] = pd.to_datetime(df["acq_date"]).dt.date

# Ensure a clean sequential row index
df = df.reset_index(drop=True)

# Stable identifier for each original VIIRS detection
df["detection_id"] = df.index

print(f"Loaded {len(df)} extreme VIIRS detections")

df.head()

## 2. Build spatial candidate pairs (≤ 600 m)

In [ ]:
# Create point geometries from the original VIIRS WGS84 coordinates
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["lon"], df["lat"]),
    crs=f"EPSG:{WGS84_EPSG}"
)

# Reproject only for efficient spatial candidate searching.
# EPSG:7755 is NOT used for the final 600 m distance decision.
gdf_search = gdf.to_crs(epsg=INDIA_CRS_EPSG)

# Extract projected X/Y coordinates in metres
coords = np.column_stack([
    gdf_search.geometry.x,
    gdf_search.geometry.y
])

# Build spatial index
tree = cKDTree(coords)

# Use a larger radius only to identify candidate pairs.
# The final spatial decision will be made using WGS84 geodesic distance.
CANDIDATE_SEARCH_RADIUS_M = 1000

spatial_candidate_pairs = tree.query_pairs(
    r=CANDIDATE_SEARCH_RADIUS_M
)

print(
    f"Spatial candidate pairs (within "
    f"{CANDIDATE_SEARCH_RADIUS_M} m projected search radius): "
    f"{len(spatial_candidate_pairs)}"
)

## 3. Apply temporal connectivity rule (Δdate ≤ 1 day)

Only spatial pairs that are also same-day or next-day are kept as true edges in the connectivity graph.

In [ ]:

# Acquisition dates for the temporal connectivity test
acq_dates = pd.to_datetime(df["acq_date"]).values.astype("datetime64[D]")

edges = []

# Extract longitude and latitude as NumPy arrays
lons = df["lon"].to_numpy()
lats = df["lat"].to_numpy()

for i, j in spatial_candidate_pairs:

    # ---------------------------------------------------------
    # 1. Calculate true WGS84 geodesic distance
    # ---------------------------------------------------------
    _, _, distance_m = GEOD.inv(
        lons[i], lats[i],
        lons[j], lats[j]
    )

    # Final spatial criterion
    spatially_connected = distance_m <= SPATIAL_THRESHOLD_M

    if not spatially_connected:
        continue

    # ---------------------------------------------------------
    # 2. Calculate temporal separation
    # ---------------------------------------------------------
    delta_days = abs(
        (acq_dates[i] - acq_dates[j])
        .astype("timedelta64[D]")
        .astype(int)
    )

    # Final spatio-temporal connectivity criterion
    if delta_days <= TEMPORAL_THRESHOLD_DAYS:
        edges.append((i, j))

print(
    f"Spatio-temporal edges "
    f"(geodesic distance <= {SPATIAL_THRESHOLD_M} m "
    f"AND temporal difference <= {TEMPORAL_THRESHOLD_DAYS} day): "
    f"{len(edges)}"
)

## 4. Connected-component clustering (assign Fire_ID)

In [ ]:

def connected_components(n_nodes, edge_list):
    """Union-Find based connected-component labeling."""
    parent = list(range(n_nodes))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(i, j):
        ri, rj = find(i), find(j)

        if ri != rj:
            parent[ri] = rj

    for i, j in edge_list:
        union(i, j)

    return np.array([find(i) for i in range(n_nodes)])


# Identify connected components from the valid
# spatio-temporal edges
raw_component_id = connected_components(
    len(df),
    edges
)

# Relabel components with clean sequential Fire_IDs
unique_components, component_labels = np.unique(
    raw_component_id,
    return_inverse=True
)

df["Fire_ID"] = [
    f"Fire_{i+1:04d}"
    for i in component_labels
]

n_events = df["Fire_ID"].nunique()

print(
    f"{len(df)} extreme detections grouped into "
    f"{n_events} unique fire events (Fire_IDs)"
)

## 5. Build the event-level catalogue

In [ ]:
# Ensure temporal fields are in the correct format
df["acq_datetime"] = pd.to_datetime(df["acq_datetime"])
df["acq_date"] = pd.to_datetime(df["acq_date"]).dt.date


def summarize_event(group):
    group = group.sort_values("acq_datetime")
    # Maximum-FRP detection
    max_frp = group["frp"].max()
    # Tie-break: if multiple detections have the same maximum FRP,
    # select the earliest acquisition record
    max_frp_row = (
        group[group["frp"] == max_frp]
        .sort_values("acq_datetime")
        .iloc[0]
    )
    # Event timing
    start_dt = group["acq_datetime"].min()
    end_dt = group["acq_datetime"].max()
    # Number of distinct days on which the event was detected
    n_active_days = group["acq_date"].nunique()
    return pd.Series({
        "basin": (
            group["basin"].iloc[0]
            if "basin" in group.columns
            else "Godavari"
        ),
        "start_datetime": start_dt,
        "end_datetime": end_dt,
        # Calendar duration including both start and end dates
        "duration_days": (
            pd.Timestamp(end_dt).normalize()
            - pd.Timestamp(start_dt).normalize()
        ).days + 1,
        "n_active_days": n_active_days,
        "n_detections": len(group),

        # Satellite information
        "n_satellites": (
            group["satellite"].nunique()
            if "satellite" in group.columns
            else np.nan
        ),
        "satellites_involved": (
            ", ".join(
                sorted(group["satellite"].dropna().unique())
            )
            if "satellite" in group.columns
            else ""
        ),
        # FRP statistics
        "mean_frp": group["frp"].mean(),
        "median_frp": group["frp"].median(),
        "min_frp": group["frp"].min(),
        "max_frp": max_frp,
        # Location and time of maximum-FRP detection
        "max_frp_datetime": max_frp_row["acq_datetime"],
        "max_frp_lat": max_frp_row["lat"],
        "max_frp_lon": max_frp_row["lon"],
        # Mean spatial location of all detections in the event
        "centroid_lat": group["lat"].mean(),
        "centroid_lon": group["lon"].mean(),
        # Dominant LULC class
        "lulc_class": (
            group["lulc_class"].mode().iloc[0]
            if (
                "lulc_class" in group.columns
                and not group["lulc_class"].mode().empty
            )
            else np.nan
        ),
    })


# Create one row per Fire_ID
event_catalogue = (
    df.groupby("Fire_ID", group_keys=True)
    .apply(summarize_event, include_groups=False)
    .reset_index()
)

print(
    f"Event-level catalogue: "
    f"{len(event_catalogue)} unique fire events"
)

event_catalogue.head()

## 6. Quality checks

In [ ]:
# --- 6a. Fire_ID count vs. original detection count -----------------------------
print(f"Original extreme detections : {len(df)}")
print(f"Unique Fire_IDs identified  : {df['Fire_ID'].nunique()}")
print(f"Reduction factor            : {len(df) / df['Fire_ID'].nunique():.2f}x detections per event (avg)")

# --- 6b. Distribution of detections-per-event (spot multi-detection events) ----
detections_per_event = df.groupby("Fire_ID").size()
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(detections_per_event, bins=range(1, detections_per_event.max() + 2), ax=ax)
ax.set_title("Detections per Fire Event")
ax.set_xlabel("Number of detections in event")
plt.tight_layout()
plt.show()

print(detections_per_event.describe())


In [ ]:
# --- 6c. Multi-satellite same-day grouping check --------------------------------
multi_sat_events = event_catalogue[event_catalogue["n_satellites"] > 1]
print(f"Events combining detections from >1 satellite: {len(multi_sat_events)} / {len(event_catalogue)}")
multi_sat_events[["Fire_ID", "n_detections", "n_satellites", "satellites_involved", "duration_days"]].head(10)


In [ ]:
# --- 6d. Multi-day events (confirms temporal chaining works) -------------------
multi_day_events = event_catalogue[event_catalogue["n_active_days"] > 1].sort_values(
    "n_active_days", ascending=False
)
print(f"Multi-day events: {len(multi_day_events)} / {len(event_catalogue)}")
multi_day_events[["Fire_ID", "n_detections", "n_active_days", "duration_days", "max_frp"]].head(10)


In [ ]:
# --- 6e. Flag unusually large / elongated events (possible over-clustering) ----

def event_spatial_extent_km(fire_id):
    pts = df.loc[
        df["Fire_ID"] == fire_id,
        ["lat", "lon"]
    ]
    if len(pts) < 2:
        return 0.0
    lons = pts["lon"].to_numpy()
    lats = pts["lat"].to_numpy()
    max_distance_m = 0.0

    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            _, _, distance_m = GEOD.inv(
                lons[i], lats[i],
                lons[j], lats[j]
            )
            max_distance_m = max(max_distance_m, distance_m)
    return max_distance_m / 1000.0


event_catalogue["max_extent_km"] = event_catalogue["Fire_ID"].apply(
    event_spatial_extent_km
)

# Flag the largest events for manual inspection
large_events = (
    event_catalogue
    .sort_values("max_extent_km", ascending=False)
    .head(10)
)

print("Largest-extent events (inspect for possible over-clustering):")
print(
    large_events[
        [
            "Fire_ID",
            "n_detections",
            "n_active_days",
            "duration_days",
            "max_extent_km"
        ]
    ]
)

In [ ]:
# --- 6f. Temporal-gap integrity spot check --------------------------------------
# For a sample multi-day event, confirm consecutive active dates have no gap > 1 day
# unless bridged by another spatially-connected detection (i.e. the algorithm's own logic).
sample_fire_id = multi_day_events["Fire_ID"].iloc[0] if len(multi_day_events) else None

if sample_fire_id:
    sample = df[df["Fire_ID"] == sample_fire_id].sort_values("acq_datetime")
    print(f"Sample multi-day event: {sample_fire_id}")
    print(sample[["acq_datetime", "lat", "lon", "frp", "satellite"]].to_string(index=False))


## 7. Visualize a selected Fire_ID for spatial validation

In [ ]:
def plot_fire_event(fire_id):
    sub = df[df["Fire_ID"] == fire_id].sort_values("acq_datetime")
    sub_gdf = gpd.GeoDataFrame(sub, geometry=gpd.points_from_xy(sub["lon"], sub["lat"]), crs="EPSG:4326")

    fig, ax = plt.subplots(figsize=(6, 6))
    sub_gdf.plot(
        ax=ax, column="acq_date", categorical=True, legend=True,
        markersize=80, edgecolor="black", cmap="viridis",
    )
    for _, row in sub.iterrows():
        ax.annotate(row["satellite"], (row["lon"], row["lat"]), fontsize=7, xytext=(3, 3), textcoords="offset points")

    ax.set_title(f"{fire_id} — {len(sub)} detections, {sub['acq_date'].nunique()} active day(s)")
    plt.tight_layout()
    plt.show()


# Visualize the largest multi-day event found above, as a representative check
if sample_fire_id:
    plot_fire_event(sample_fire_id)


## 8. Save final outputs

In [ ]:
# A. Detection-level dataset with Fire_ID
detection_out_path = OUT_DIR / "extreme_agri_fire_detections_with_fireid_godavari_2019_2024.csv"
df.drop(columns="detection_id").to_csv(detection_out_path, index=False)
print("Saved:", detection_out_path)

# B. Event-level unique fire catalogue
catalogue_out_path = OUT_DIR / "unique_fire_event_catalogue_godavari_2019_2024.csv"
event_catalogue.to_csv(catalogue_out_path, index=False)
print("Saved:", catalogue_out_path)


In [ ]:
# Find the Fire_ID with the maximum active days
max_active_row = event_catalogue.loc[event_catalogue["n_active_days"].idxmax()]

print(f"Fire ID with max active days : {max_active_row['Fire_ID']}")
print(f"Number of active days        : {max_active_row['n_active_days']}")
print(f"Total duration (days)        : {max_active_row['duration_days']}")